In [1]:
import os
import time
import numpy as np
from tqdm import tqdm

import pybullet as p
import pybullet_data
from surrol.utils.pybullet_utils import (
    step,
    get_joints,
    get_link_name,
    reset_camera,
)
from surrol.robots.psm import Psm

p.connect(p.GUI)
# p.connect(p.DIRECT)
p.setGravity(0, 0, -9.81)

POSE_TRAY = ((0.55, 0, 0.6781), (0, 0, 0))
SCALING = 5.
workspace_limits = np.array([(2.5, 3), (-0.25, 0.25), (3.426, 3.776)])

tray_id = p.loadURDF('/Users/kantaphat/Research/DEX/SurRoL/surrol/assets/tray/tray.urdf',
                    np.array(POSE_TRAY[0]) * SCALING,
                    p.getQuaternionFromEuler(POSE_TRAY[1]),
                    globalScaling=SCALING)
p.changeVisualShape(tray_id, -1, rgbaColor=(225 / 255, 225 / 255, 225 / 255, 1))

reset_camera(yaw=90, pitch=-40, dist=1, target=np.array(POSE_TRAY[0]) * SCALING)

POSE_PSM1 = ((0.05, 0.24, 0.8524), (0, 0, -(90 + 20) / 180 * np.pi))
psm = Psm(POSE_PSM1[0], 
          p.getQuaternionFromEuler(POSE_PSM1[1]), 
          scaling=SCALING)

pybullet build time: Feb 15 2025 12:06:37


Version = 4.1 Metal - 89.4
Vendor = Apple
Renderer = Apple M1
b3Printf: Selected demo: Physics Server
startThreads creating 1 threads.
starting thread 0
started thread 0 
MotionThreadFunc thread started


In [3]:
pos = (workspace_limits[0][0],
       workspace_limits[1][1],
      (workspace_limits[2][1] + workspace_limits[2][0]) / 2)
orn = (0.5, 0.5, -0.5, -0.5)
joint_positions = psm.inverse_kinematics((pos, orn), psm.EEF_LINK_INDEX)
psm.reset_joint(joint_positions)

array([ 0.18549925, -0.00715823,  0.14099653, -0.34339686, -0.0563831 ,
       -0.17695996])

In [14]:
cyl_radius = 0.1
cyl_length = 0.15
cyl_pos = (
    workspace_limits[0].mean(),
    workspace_limits[1].mean(),
    workspace_limits[2][0] + 0.045
)

cylinder_id = p.createMultiBody(
    baseMass=0,
    baseVisualShapeIndex=p.createVisualShape(
        p.GEOM_CYLINDER, 
        radius=cyl_radius, 
        length=cyl_length, 
        rgbaColor=[0, 1, 0, 0.3]
    ),
    basePosition=cyl_pos,
    baseOrientation=p.getQuaternionFromEuler([0, 0, 0])
)

In [13]:
p.removeBody(cylinder_id)

In [5]:
reset_camera(yaw=90, pitch=-30, dist=0.6, target=np.array(POSE_TRAY[0]) * SCALING)

In [6]:
def draw_workspace_box(limits, color=[0, 0, 1], line_width=1, lifetime=0):
    """
    Draw a bounding box for workspace limits in PyBullet
    
    Args:
        limits: ((x_min, x_max), (y_min, y_max), (z_min, z_max))
        color: [r, g, b] line color
        line_width: width of the lines
        lifetime: 0 for permanent, >0 for temporary in seconds
    """
    x_min, x_max = limits[0]
    y_min, y_max = limits[1] 
    z_min, z_max = limits[2]
    
    # Define the 8 corners of the box
    corners = [
        [x_min, y_min, z_min], [x_max, y_min, z_min],
        [x_min, y_max, z_min], [x_max, y_max, z_min],
        [x_min, y_min, z_max], [x_max, y_min, z_max],
        [x_min, y_max, z_max], [x_max, y_max, z_max]
    ]
    
    # Define the 12 edges of the box
    edges = [
        # Bottom face (z_min)
        (0, 1), (0, 2), (1, 3), (2, 3),
        # Top face (z_max)
        (4, 5), (4, 6), (5, 7), (6, 7),
        # Vertical edges
        (0, 4), (1, 5), (2, 6), (3, 7)
    ]
    
    # Draw all edges
    line_ids = []
    for edge in edges:
        start_pos = corners[edge[0]]
        end_pos = corners[edge[1]]
        line_id = p.addUserDebugLine(start_pos, end_pos, color, line_width, lifetime)
        line_ids.append(line_id)
    
    return line_ids

# Remove the box later
def remove_workspace_box(line_ids):
    """Remove previously drawn workspace box"""
    for line_id in line_ids:
        p.removeUserDebugItem(line_id)

In [7]:
workspace_id = draw_workspace_box(workspace_limits, color=[0, 0, 1])

In [ ]:
remove_workspace_box(workspace_id)

In [8]:
original_gauze_ranges = np.array((
    (workspace_limits[0].mean() - 0.05, workspace_limits[0].mean() + 0.05),
    (workspace_limits[1].mean() - 0.05, workspace_limits[1].mean() + 0.05),
    (workspace_limits[2][0] + 0.01, workspace_limits[2][0] + 0.009)
))

gauze_ranges = np.array((
    (workspace_limits[0].mean() - 0.05, workspace_limits[0].mean() + 0.05),
    (workspace_limits[1][0], workspace_limits[1][0] + 0.08),
    (workspace_limits[2][0] + 0.01, workspace_limits[2][0] + 0.009)
))

In [12]:
gauze_ranges_id = draw_workspace_box(gauze_ranges, color=[1, 0, 0])
original_gauze_ranges_id = draw_workspace_box(original_gauze_ranges, color=[0, 1, 0])

In [11]:
remove_workspace_box(gauze_ranges_id)
remove_workspace_box(original_gauze_ranges_id)

In [ ]:
def get_scaled_obb(obj_path, pybullet_scale):
    # Load vertices from OBJ and apply PyBullet scale
    vertices = []
    with open(obj_path) as f:
        for line in f:
            if line.startswith("v "):
                vertex = np.array([float(x) for x in line.split()[1:4]])
                vertex_scaled = vertex * pybullet_scale  # Apply scale
                vertices.append(vertex_scaled)
    vertices = np.array(vertices)
    
    # Compute OBB bounds
    return vertices.min(axis=0), vertices.max(axis=0)

obj_path = '/Users/kantaphat/Research/DEX/SurRoL/surrol/assets/gauze/meshes/gauze.obj'
aabb_min, aabb_max = get_scaled_obb(obj_path, SCALING)
gauze_dimensions = np.array(aabb_max) - np.array(aabb_min)
gauze_dimensions

In [ ]:
gauze_dimensions[1] / 2

In [26]:
gauze_id = p.loadURDF('/Users/kantaphat/Research/DEX/SurRoL/surrol/assets/gauze/gauze.urdf',
                    (np.random.uniform(gauze_ranges[0][0], gauze_ranges[0][1]),
                    np.random.uniform(gauze_ranges[1][0], gauze_ranges[1][1]),
                    # gauze_ranges[1][1],
                    workspace_limits[2][0] + 0.01),
                    (0, 0, 0, 1),
                    useFixedBase=False,
                    globalScaling=SCALING)

p.changeVisualShape(gauze_id, -1, specularColor=(0, 0, 0))

In [25]:
p.removeBody(gauze_id)

In [ ]:
(workspace_limits[1].mean() - workspace_limits[1][0] - 0.1 * 2)

In [19]:
original_goal_ranges = np.array((
    (workspace_limits[0].mean() + 0.02 * -2.576 * SCALING, workspace_limits[0].mean() + 0.02 * 2.576 * SCALING),
    (workspace_limits[1].mean() + 0.02 * -2.576 * SCALING, workspace_limits[1].mean() + 0.02 * 2.576 * SCALING),
    (workspace_limits[2][1] - 0.03 * SCALING, workspace_limits[2][1] - 0.029 * SCALING)
))

goal_ranges = np.array((
    (workspace_limits[0].mean() + 0.01 * -2.576 * SCALING, workspace_limits[0].mean() + 0.01 * 2.576 * SCALING),
    (workspace_limits[1].mean() + 0.01 * SCALING * 2.576, workspace_limits[1][1]),
    (workspace_limits[2][1] - 0.03 * SCALING, workspace_limits[2][1] - 0.029 * SCALING)
))

In [20]:
goal_id = draw_workspace_box(goal_ranges, color=[0, 0, 0])
original_goal_id = draw_workspace_box(original_goal_ranges, color=[0, 1, 1])

In [18]:
remove_workspace_box(goal_id)
remove_workspace_box(original_goal_id)

In [27]:
while True:
    p.stepSimulation()
    time.sleep(1.0/240.0)

KeyboardInterrupt: 